# 05 - Transformer with Defense 3 (Differential Privacy)

Defense: DP-SGD with epsilon target 1.0 or 5.0

In [1]:
import os, random
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow_privacy.privacy.optimizers.dp_optimizer_keras import DPKerasAdamOptimizer

In [5]:
EPSILON_TARGET = 5.0  # 1.0 or 5.0
BATCH_SIZE = 32
NUM_MICROBATCHES = 1
NOISE_MULTIPLIER = 1.9 if EPSILON_TARGET <= 1.0 else 1.1

BASELINE_TEST_ACC = 0.9331
BASELINE_ATTACK = pd.DataFrame([
    {'attack':'threshold_loss','auc':0.4821,'accuracy':0.3099,'f1':0.3636},
    {'attack':'logistic','auc':0.4818,'accuracy':0.8028,'f1':0.0},
    {'attack':'shadow_meta','auc':0.6151,'accuracy':0.6485,'f1':0.6802},
])

def set_seed(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed); np.random.seed(seed); tf.keras.utils.set_random_seed(seed)

def build_dp_transformer(n_features):
    inp = layers.Input(shape=(1, n_features))
    x = layers.Dense(64)(inp)
    a = layers.MultiHeadAttention(num_heads=4, key_dim=16, dropout=0.15)(x, x)
    x = layers.LayerNormalization(epsilon=1e-6)(x + a)
    f = layers.Dense(128, activation='relu')(x)
    f = layers.Dropout(0.15)(f)
    f = layers.Dense(64)(f)
    x = layers.LayerNormalization(epsilon=1e-6)(x + f)
    x = layers.Flatten()(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.15)(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    m = Model(inp, out)
    opt = DPKerasAdamOptimizer(l2_norm_clip=1.0, noise_multiplier=NOISE_MULTIPLIER, num_microbatches=NUM_MICROBATCHES, learning_rate=1e-3)
    loss = tf.keras.losses.BinaryCrossentropy(reduction=tf.keras.losses.Reduction.NONE)
    m.compile(optimizer=opt, loss=loss, metrics=['accuracy'])
    return m

def mia_features(p, y):
    p = np.clip(np.asarray(p), 1e-8, 1-1e-8); y = np.asarray(y)
    loss = -(y*np.log(p)+(1-y)*np.log(1-p))
    conf = np.maximum(p,1-p); ent = -(p*np.log(p)+(1-p)*np.log(1-p))
    cor = ((p>=0.5).astype(int)==y).astype(float); margin = np.abs(p-0.5)
    return np.column_stack([loss,conf,ent,cor,margin])

def attack_row(name, y_true, y_pred, y_score):
    return {'attack':name,'auc':roc_auc_score(y_true,y_score),'accuracy':accuracy_score(y_true,y_pred),
            'precision':precision_score(y_true,y_pred,zero_division=0),'recall':recall_score(y_true,y_pred,zero_division=0),
            'f1':f1_score(y_true,y_pred,zero_division=0)}

def run_attacks(model, Xtr_seq, ytr, Xte_seq, yte, Xall, yall, seed):
    p_tr = model.predict(Xtr_seq, verbose=0).ravel(); p_te = model.predict(Xte_seq, verbose=0).ravel()
    Fm, Fn = mia_features(p_tr,ytr), mia_features(p_te,yte)
    Xa = np.vstack([Fm,Fn]); ya = np.concatenate([np.ones(len(Fm),dtype=int), np.zeros(len(Fn),dtype=int)])
    Xa_tr, Xa_te, ya_tr, ya_te = train_test_split(Xa, ya, test_size=0.4, random_state=seed, stratify=ya)

    score_tr = -Xa_tr[:,0]; score_te = -Xa_te[:,0]
    thrs = np.linspace(score_tr.min(), score_tr.max(), 300)
    bt, bb = thrs[0], -1
    for t in thrs:
        pred = (score_tr >= t).astype(int)
        tpr = ((pred==1)&(ya_tr==1)).sum()/max((ya_tr==1).sum(),1)
        tnr = ((pred==0)&(ya_tr==0)).sum()/max((ya_tr==0).sum(),1)
        b = 0.5*(tpr+tnr)
        if b > bb: bb, bt = b, t
    r_thr = attack_row('threshold_loss', ya_te, (score_te>=bt).astype(int), score_te)

    lr = LogisticRegression(max_iter=1000, random_state=seed); lr.fit(Xa_tr, ya_tr)
    plr = lr.predict_proba(Xa_te)[:,1]
    r_lr = attack_row('logistic', ya_te, (plr>=0.5).astype(int), plr)

    Xab, Xaux, yab, yaux = train_test_split(Xall, yall, test_size=0.30, random_state=seed, stratify=yall)
    Xm, Xn, ym, yn = train_test_split(Xab, yab, test_size=0.45, random_state=seed+1, stratify=yab)
    sct = StandardScaler(); Xm_sc = sct.fit_transform(Xm).astype(np.float32); Xn_sc = sct.transform(Xn).astype(np.float32)
    Xm_seq = Xm_sc.reshape(-1,1,Xm_sc.shape[1]); Xn_seq = Xn_sc.reshape(-1,1,Xn_sc.shape[1])
    set_seed(seed+1000); tf.keras.backend.clear_session()
    tgt = build_dp_transformer(Xm_sc.shape[1]); tgt.fit(Xm_seq, ym, epochs=50, batch_size=BATCH_SIZE, verbose=0)
    Xt = np.vstack([mia_features(tgt.predict(Xm_seq, verbose=0).ravel(), ym), mia_features(tgt.predict(Xn_seq, verbose=0).ravel(), yn)])
    yt = np.concatenate([np.ones(len(ym),dtype=int), np.zeros(len(yn),dtype=int)])

    SX, SY = [], []
    for i in range(8):
        xsm, xsn, ysm, ysn = train_test_split(Xaux, yaux, test_size=0.5, random_state=seed+100+i, stratify=yaux)
        scs = StandardScaler(); xsm_sc = scs.fit_transform(xsm).astype(np.float32); xsn_sc = scs.transform(xsn).astype(np.float32)
        xsm_seq = xsm_sc.reshape(-1,1,xsm_sc.shape[1]); xsn_seq = xsn_sc.reshape(-1,1,xsn_sc.shape[1])
        set_seed(seed+2000+i); tf.keras.backend.clear_session()
        sh = build_dp_transformer(xsm_sc.shape[1]); sh.fit(xsm_seq, ysm, epochs=30, batch_size=BATCH_SIZE, verbose=0)
        SX.append(np.vstack([mia_features(sh.predict(xsm_seq, verbose=0).ravel(), ysm), mia_features(sh.predict(xsn_seq, verbose=0).ravel(), ysn)]))
        SY.append(np.concatenate([np.ones(len(ysm),dtype=int), np.zeros(len(ysn),dtype=int)]))

    meta = GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=seed)
    meta.fit(np.vstack(SX), np.concatenate(SY))
    sc = meta.predict_proba(Xt)[:,1]
    r_sh = attack_row('shadow_meta', yt, (sc>=0.5).astype(int), sc)
    return pd.DataFrame([r_thr,r_lr,r_sh]).sort_values('auc', ascending=False)

In [3]:
print('=== Load prepared data ===')
b = np.load('../data_preparation/prepared_oasis2_transformer.npz', allow_pickle=True)
X, y = b['X'], b['y']; X_train, X_test = b['X_train'], b['X_test']
y_train, y_test = b['y_train'], b['y_test']; X_train_seq, X_test_seq = b['X_train_seq'], b['X_test_seq']
SEED = int(b['seed'][0])
print(f'X={X.shape}, train={X_train.shape}, test={X_test.shape}, eps_target={EPSILON_TARGET}')

=== Load prepared data ===
X=(354, 9), train=(70, 9), test=(284, 9), eps_target=5.0


In [6]:
print('=== Train DP-SGD model ===')
set_seed(SEED + 10); tf.keras.backend.clear_session()
model_dp = build_dp_transformer(X_train.shape[1])
es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=0)
h = model_dp.fit(X_train_seq, y_train, epochs=90, batch_size=BATCH_SIZE, validation_split=0.2, callbacks=[es], verbose=0)
p_te = model_dp.predict(X_test_seq, verbose=0).ravel()
test_acc_def3 = accuracy_score(y_test, (p_te>=0.5).astype(int))
print(f'test_acc_def3={test_acc_def3:.4f}, epochs={len(h.history["loss"])}')

=== Train DP-SGD model ===
test_acc_def3=0.5810, epochs=90


In [7]:
print('=== Run MIA attacks (Defense 3) ===')
attack_def3 = run_attacks(model_dp, X_train_seq, y_train, X_test_seq, y_test, X, y, SEED)
display(attack_def3.round(4))

=== Run MIA attacks (Defense 3) ===


,attack,auc,accuracy,precision,recall,f1
1,logistic,0.5852,0.8028,0.0000,0.0000,0.0000
2,shadow_meta,0.5754,0.5709,0.6074,0.6074,0.6074
0,threshold_loss,0.5721,0.6831,0.1600,0.1429,0.1509


In [8]:
print('=== Baseline vs Defense 3 ===')
cmp = BASELINE_ATTACK.merge(attack_def3[['attack','auc','accuracy','f1']], on='attack', suffixes=('_baseline','_defense3'))
cmp['delta_auc'] = cmp['auc_defense3'] - cmp['auc_baseline']
cmp['delta_accuracy'] = cmp['accuracy_defense3'] - cmp['accuracy_baseline']
cmp['delta_f1'] = cmp['f1_defense3'] - cmp['f1_baseline']
display(cmp.sort_values('attack').round(4))

mcmp = pd.DataFrame([{'model':'Transformer','test_acc_baseline':BASELINE_TEST_ACC,'test_acc_defense3':test_acc_def3,'delta_test_acc':test_acc_def3-BASELINE_TEST_ACC,'epsilon_target':EPSILON_TARGET}])
display(mcmp.round(4))

=== Baseline vs Defense 3 ===


,attack,auc_baseline,accuracy_baseline,f1_baseline,auc_defense3,accuracy_defense3,f1_defense3,delta_auc,delta_accuracy,delta_f1
1,logistic,0.4818,0.8028,0.0000,0.5852,0.8028,0.0000,0.1034,0.0000,0.0000
2,shadow_meta,0.6151,0.6485,0.6802,0.5754,0.5709,0.6074,-0.0397,-0.0776,-0.0728
0,threshold_loss,0.4821,0.3099,0.3636,0.5721,0.6831,0.1509,0.0900,0.3732,-0.2127


,model,test_acc_baseline,test_acc_defense3,delta_test_acc,epsilon_target
0,Transformer,0.9331,0.581,-0.3521,5.0


In [9]:
print('=== Quick AUC summary ===')
quick_auc = cmp[['attack','auc_baseline','auc_defense3','delta_auc']].sort_values('delta_auc')
display(quick_auc.round(4))

=== Quick AUC summary ===


,attack,auc_baseline,auc_defense3,delta_auc
2,shadow_meta,0.6151,0.5754,-0.0397
0,threshold_loss,0.4821,0.5721,0.0900
1,logistic,0.4818,0.5852,0.1034
